[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/geraldmc/irilab2026/blob/main/notebooks/r2/r2-q2/04_categorization.ipynb)

# R2-Q2 Week 4 — Apply the taxonomy and report the failure-mode distribution

This notebook applies the four numeric predicates committed in Week 1 to the Grad-CAM heatmaps produced in Week 3, assigning each misclassification to one of five categories — symptom-attended-but-wrong-class, background-attended, lighting-attended, leaf-shape-attended, or other/ambiguous — following the first-match-wins scheme with multi-fires routing to "other." The output is the failure-mode distribution across the misclassification set, reported at the overall taxonomy level rather than per-class, and the methods-section text documenting how each predicate was operationalized.

By the end of this notebook you will have:

- A `categorizations.parquet` containing one row per misclassification with columns recording which predicates fired, the assigned category, and the derived quantities (m_out, m_perimeter, lighting_correlation, m_in, concentration) that drove the assignment — so a reader can audit any individual categorization.
- A `taxonomy_distribution.json` recording the count and fraction of misclassifications in each of the five categories, the conjunction-handling statistics (how many images fired multiple predicates and routed to "other"), and the headline interpretation sentence describing what the failure-mode distribution looks like.
- A `taxonomy_distribution.png` bar chart showing the five-category distribution with the "other" bucket visually distinguished — so a reader can see at a glance whether failures clustered into recognizable patterns or whether the taxonomy itself needs revision.

## Before you run anything: switch to a GPU runtime

This notebook uses a large vision model (the Segment Anything Model,
or SAM) to separate the leaf from the background in each of the 81
misclassified images. SAM runs much faster on a GPU than on a CPU,
and 81 images on CPU adds up to a long wait. Colab's default runtime
is CPU-only, so you'll need to switch.

To switch:

1. In Colab's menu bar: **Runtime → Change runtime type**.
2. Under "Hardware accelerator," choose **T4 GPU**.
3. Click **Save**.

Wait for the session to come up on the new runtime (a few seconds).
Then run the cells below in order.

**If Colab tells you "no GPUs available right now":** free-tier GPU
access is shared and occasionally unavailable. Wait a few minutes
and try again. If it keeps refusing, tell your mentor — the
upgraded Colab tiers have more reliable GPU access.

In [ ]:
try:
    import irilab2026 as iri
    print(f"OK — irilab2026 {iri.__version__} is installed and all deps importable.")
    print("    → use Pattern 2 in the following Cell:")
    print("      !pip install --force-reinstall --no-deps git+https://github.com/geraldmc/irilab2026.git@main")
except ModuleNotFoundError as e:
    print(f"MISSING — {e.name} not importable.")
    print("    → use Pattern 1 in the following Cell:")
    print("      !pip install git+https://github.com/geraldmc/irilab2026.git@main")

In [ ]:
# Pattern 1 — fresh or partial runtime (installs deps that aren't present yet)
# This skips anything pip already sees as installed. This is ideal for a new environment or when you want
# to be sure everything is up to date.

# !pip install git+https://github.com/geraldmc/irilab2026.git@main

# Pattern 2 — populated runtime (refreshes irilab2026 only, leaves deps alone)
# This is ideal for when you already have the dependencies installed and just want to update irilab2026.

# !pip install --force-reinstall --no-deps git+https://github.com/geraldmc/irilab2026.git@main

### Confirm the GPU is in place

Before going further, confirm Colab attached a GPU. The cell below
prints `GPU detected:` followed by `True` or `False`. If it's `True`,
you're good. If it's `False`, go back to the top of the notebook and
re-do the runtime switch — the next cells will not work without a GPU.

The check is a separate cell from the main setup (below) because it's
fast and recoverable: if the GPU isn't there, you see the problem
right away with no Drive authorization prompt to dismiss first.

In [ ]:
import irilab2026 as iri

gpu_ok = iri.has_gpu()
print(f"GPU detected: {gpu_ok}")

assert gpu_ok, (
    "No GPU detected. Go to the top of the notebook and follow the "
    "runtime-switch instructions, then re-run from Setup Step 1."
)

In [ ]:
# Mount Drive and set up irilab2026 with a GPU.
import irilab2026 as iri
iri.setup(gpu_required=True)

OUTPUT_DIR = iri.output_dir("r2_q2")
print(f"Output directory: {OUTPUT_DIR}")

## 1) Load the inputs from the earlier notebooks

This notebook reads three files produced by the earlier notebooks in
this series:

- `precommit.json`, from `01_pre_commits.ipynb`
- `working_set.parquet`, from `02_load_and_filter.ipynb`
- `gradcam_metadata.parquet`, from `03_sanity_checks_and_gradcam.ipynb`

The three subsections below load each in turn, with the validation
needed to catch the most common ways they can be missing or wrong.
All three loads share the same posture: try to read the file, raise
a helpful error if it's missing, validate just enough to catch the
obvious failures, and print what got loaded so you can see the
structure without inspecting it manually.

Deeper validation of the precommit's internal structure happens in
Sections 3, 4, and 5 — at the point where each sub-field is actually
used. That keeps error messages close to the symptoms that produce
them.

### 1.1) Load and validate the pre-commit

The precommit is the contract from Week 1 — the document that captures
every decision about how this categorization will run. N04 reads from
it in every later section: the leaf-segmentation method (Section 3),
the five derived quantities (Section 4), the four predicates with
their thresholds and order (Section 5), and the aggregation level for
the final summary (Section 6).

The validation below is intentionally light. It confirms the four
top-level fields the precommit promises (`taxonomy`,
`categorization_procedure`, `aggregation_level`, `sanity_checks`) —
enough to say "this is a real precommit, not an empty file" — plus
one slightly deeper check that the taxonomy's category list is
populated, since every later section iterates over it. Validation
of specific sub-fields (the thresholds, the predicate order, the
segmentation parameters) happens in the section that reads each one.

In [ ]:
### 1.1) Load and validate the pre-commit

import json

precommit_path = OUTPUT_DIR / "precommit.json"

try:
    with open(precommit_path) as f:
        precommit = json.load(f)
except FileNotFoundError:
    raise FileNotFoundError(
        f"Could not find {precommit_path}. "
        "This file is produced by 01_pre_commits.ipynb — "
        "run that notebook to completion before running this one."
    ) from None

# Light validation: the four locked top-level keys must all be present.
# Each later section validates the specific sub-fields it uses.
expected_keys = {
    "taxonomy",
    "categorization_procedure",
    "sanity_checks",
    "aggregation_level",
}
missing = expected_keys - set(precommit.keys())
if missing:
    raise KeyError(
        f"precommit.json is missing top-level keys: {sorted(missing)}. "
        "This usually means N01 didn't complete — "
        "re-run 01_pre_commits.ipynb to regenerate the file."
    )

# Slightly deeper: the taxonomy categories list is what every later
# section iterates over. Check here that it's present and non-empty
# so the print below doesn't crash with a noisy KeyError.
if not precommit["taxonomy"].get("categories"):
    raise KeyError(
        "precommit['taxonomy']['categories'] is missing or empty. "
        "N01 Section 4 didn't complete — re-run 01_pre_commits.ipynb."
    )

print(f"Loaded: {precommit_path}")
print(f"  top-level keys: {sorted(precommit.keys())}")
print(f"  taxonomy ({len(precommit['taxonomy']['categories'])} categories):")
for cat in precommit["taxonomy"]["categories"]:
    print(f"    - {cat}")

### 1.2) Load the working set

N02 wrote `working_set.parquet` to this question's output directory:
the rows of the PlantDoc prediction table where the model
misclassified at the category level. N03 read this same file to
generate the Grad-CAM heatmaps; N04 reads it again to pair each
misclassification with its heatmap and its source image.

This notebook uses one column from the working set: `filename`. It
serves two purposes — joining each row to its heatmap in the next
subsection, and looking up the source image from the PlantDoc HF
Dataset in Section 3 (the same approach N03 used to feed images
into Grad-CAM). All the other columns (`true_category`, `host`,
`disease`, etc.) pass through to the per-image output table N04
produces in Section 6, so a reader can audit any individual
categorization against the original prediction data.

In [ ]:
### 1.2) Load the working set

import pandas as pd

working_set_path = OUTPUT_DIR / "working_set.parquet"

try:
    working_set = pd.read_parquet(working_set_path)
except FileNotFoundError:
    raise FileNotFoundError(
        f"Could not find {working_set_path}. "
        "This file is produced by 02_load_and_filter.ipynb — "
        "run that notebook to completion before running this one."
    ) from None

# Light schema validation: the one column N04 strictly depends on.
# `filename` is the join key for the heatmap metadata and also the
# lookup key into the PlantDoc HF Dataset that Sections 3 and 4 use
# to pull source images.
required_columns = {"filename"}
missing = required_columns - set(working_set.columns)
if missing:
    raise KeyError(
        f"working_set.parquet is missing required columns: {sorted(missing)}. "
        "This usually means N02 wrote an older or partial schema — "
        "re-run 02_load_and_filter.ipynb to regenerate the file."
    )

# An empty working set means there are no misclassifications to
# categorize — methodologically interesting but operationally a stop
# condition for N04.
if len(working_set) == 0:
    raise ValueError(
        "working_set.parquet has zero rows. "
        "There are no misclassifications to categorize — "
        "if this is unexpected, check N02's filter logic."
    )

print(f"Loaded: {working_set_path}")
print(f"  rows: {len(working_set)}")
print(f"  columns: {sorted(working_set.columns)}")

### 1.3) Load the Grad-CAM metadata

N03 wrote one `.npy` file per misclassification — 81 heatmaps at 7×7
resolution, stored under `OUTPUT_DIR / "heatmaps"`. Alongside them
N03 wrote `gradcam_metadata.parquet`, a table that joins each heatmap
file back to its working-set row. Section 4 will load each heatmap
as it processes the corresponding image; this section confirms the
metadata is well-formed and the files are where the metadata says
they are.

The validation here is more aggressive than the previous two loads
because the file-existence check is genuinely cheap — 81 stat calls
take well under a second — and catching a missing heatmap here
saves the better part of a minute of SAM segmentation in Section 3
before the error would otherwise surface in Section 4.

Three checks, in order:

1. Every required column is present.
2. Every working-set row has a matching entry in the metadata.
3. Every referenced `.npy` file actually exists on disk.

In [ ]:
### 1.3) Load the Grad-CAM metadata

from pathlib import Path

gradcam_metadata_path = OUTPUT_DIR / "gradcam_metadata.parquet"

try:
    gradcam_metadata = pd.read_parquet(gradcam_metadata_path)
except FileNotFoundError:
    raise FileNotFoundError(
        f"Could not find {gradcam_metadata_path}. "
        "This file is produced by 03_sanity_checks_and_gradcam.ipynb — "
        "run that notebook to completion before running this one."
    ) from None

if len(gradcam_metadata) == 0:
    raise ValueError(
        f"{gradcam_metadata_path} has zero rows. "
        "Re-run 03_sanity_checks_and_gradcam.ipynb to regenerate the file."
    )

# Check 1: required columns.
required_columns = {"filename", "heatmap_path"}
missing = required_columns - set(gradcam_metadata.columns)
if missing:
    raise KeyError(
        f"gradcam_metadata.parquet is missing required columns: "
        f"{sorted(missing)}. This usually means N03 wrote an older or "
        "partial schema — re-run 03_sanity_checks_and_gradcam.ipynb "
        "to regenerate the file."
    )

# Check 2: every working-set row has a matching heatmap entry.
working_filenames = set(working_set["filename"])
heatmap_filenames = set(gradcam_metadata["filename"])
missing_entries = working_filenames - heatmap_filenames
if missing_entries:
    raise KeyError(
        f"{len(missing_entries)} working-set row(s) have no entry in "
        f"gradcam_metadata. First few: {sorted(missing_entries)[:3]}. "
        "Re-run 03_sanity_checks_and_gradcam.ipynb to regenerate the "
        "metadata."
    )

# Check 3: every referenced .npy file exists on disk. Paths may be
# stored as absolute or relative; resolve relative paths against
# OUTPUT_DIR before checking.
missing_files = []
for p in gradcam_metadata["heatmap_path"]:
    resolved = Path(p)
    if not resolved.is_absolute():
        resolved = OUTPUT_DIR / resolved
    if not resolved.exists():
        missing_files.append(str(resolved))

if missing_files:
    raise FileNotFoundError(
        f"{len(missing_files)} heatmap file(s) referenced in "
        f"gradcam_metadata are missing from disk. "
        f"First few: {missing_files[:3]}. "
        "Re-run 03_sanity_checks_and_gradcam.ipynb to regenerate the "
        "heatmaps."
    )

print(f"Loaded: {gradcam_metadata_path}")
print(f"  rows: {len(gradcam_metadata)}")
print(f"  all {len(working_set)} working-set rows have matching heatmaps — OK")
print(f"  all {len(gradcam_metadata)} heatmap files present on disk — OK")

## 2) Segment the leaf in each image

Sections 3 and 4 need to know which pixels in each image are leaf and
which are not. The five derived quantities all reference the leaf
region — `m_out` is the fraction of attention mass that falls outside
the leaf, `m_perimeter` is the fraction in a band near the leaf's edge,
and so on. Without a mask there's no in-or-out distinction and no
derived quantity to compare against the precommit's thresholds.

This section produces that mask for every image in the working set.
The precommit names SAM (the Segment Anything Model) as the
segmentation method, with the prompting strategy "single point at
image center" and a fallback to color-based thresholding if SAM
proves unreliable. Reliability is checked against five hand-segmented
reference masks shipped with the program.

The five subsections:

| Sub | What it does |
|---|---|
| 2.1 | Set up SAM — install, download the checkpoint, build the predictor, define `segment_leaf` |
| 2.2 | Run `segment_leaf` on all 81 working-set images and save raw SAM masks |
| 2.3 | Validate against five hand-segmented reference masks — compute IoU per image |
| 2.4 | Apply the pass-or-fallback decision per the precommit's criterion |
| 2.5 | Save the final per-image masks as PNGs for Section 3 |

By the end of this section, `OUTPUT_DIR / "leaf_masks"` will contain 81
PNG files, one per working-set image, plus a `segmentation_summary.json`
recording the IoU statistics, the pass/fail verdict, and which images
(if any) ended up using the color fallback rather than SAM.

### 2.1) Set up SAM

This subsection assembles the machinery the rest of Section 3 will
use: the SAM model loaded on GPU, a PlantDoc image loader (the same
shape as N03's, but returning the raw PIL image rather than a
normalized tensor), and a `segment_leaf` helper that wraps SAM's
predict API to match the precommit's prompting strategy.

A few things worth knowing upfront:

**Why `vit_h`.** SAM ships in three sizes (`vit_b`, `vit_l`, `vit_h`).
`vit_h` is the largest and most accurate, especially on cluttered
field-condition backgrounds — which is exactly what PlantDoc images
have. The cost is a one-time 2.4 GB download per Drive (cached
afterward) and a few hundred milliseconds of inference per image on
a T4 GPU.

**The checkpoint is cached in your Drive.** First call downloads the
file into `OUTPUT_DIR`; subsequent runtimes pick it up from there.
Restarting the Colab runtime doesn't trigger another download.

**`segment_leaf` returns a boolean mask.** Same height and width as
the input image, `True` where SAM identifies the leaf. The prompting
strategy is the one committed in N01: a single foreground point at
the image center, with SAM's three candidate masks scored internally
and the highest-scoring one returned.

The smoke test at the end runs `segment_leaf` on the first working-set
image and plots the result, so you can see SAM is working before
committing to the full 81-image run in 3.2.

In [ ]:
### 2.1) Install segment-anything

# segment-anything is not a dependency of irilab2026 (it's only used
# in this notebook), so we pip-install it here. On a fresh runtime
# this takes ~10 seconds; on a warm runtime pip recognizes it's
# already installed and exits immediately.

!pip install -q segment-anything

In [ ]:
### 2.2) Load PlantDoc and define the image loader

# Load PlantDoc metadata and image data. First call on a fresh runtime
# downloads ~950 MB from Hugging Face and caches to Drive; subsequent
# calls hit the cache. The metadata DataFrame has a `filename` column
# that matches the working set's join key.
pd_metadata, pd_hf_dataset = iri.load_plantdoc()
print(f"PlantDoc loaded: {len(pd_metadata)} rows")

# Build a filename → pandas-index lookup so we can pull a specific
# image by its filename in O(1). The HF dataset is index-aligned to
# the metadata, so the pandas index doubles as the hf_dataset position.
filename_to_pd_index = dict(zip(pd_metadata["filename"], pd_metadata.index))


def load_pd_image(filename):
    """Load a PlantDoc image by filename, returning the raw PIL image
    at the original resolution.

    N04 needs the raw image rather than N03's eval-transformed tensor
    because SAM and the brightness analysis in Section 4 both work on
    the full-resolution RGB pixels, not 224×224 normalized tensors.
    """
    if filename not in filename_to_pd_index:
        raise KeyError(
            f"{filename} not found in PlantDoc metadata. "
            "Check that the working set was built against the same "
            "PD dataset version."
        )
    pd_idx = filename_to_pd_index[filename]
    image_pil = pd_hf_dataset[int(pd_idx)]["image"]
    return image_pil.convert("RGB")


# Quick loader smoke test — confirm it returns a PIL image of plausible size.
_first = working_set["filename"].iloc[0]
_test = load_pd_image(_first)
print(f"Loader smoke test: {_first} → PIL image, size {_test.size}, mode {_test.mode}")

In [ ]:
### 2.3) Download the SAM checkpoint and build the predictor

import torch
from segment_anything import sam_model_registry, SamPredictor

SAM_CHECKPOINT_URL = "https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth"
SAM_CHECKPOINT_PATH = OUTPUT_DIR / "sam_vit_h_4b8939.pth"

if not SAM_CHECKPOINT_PATH.exists():
    print(f"Downloading SAM vit_h checkpoint to {SAM_CHECKPOINT_PATH}")
    print("This is a one-time download (~2.4 GB). Subsequent runtimes hit the cache.")
    torch.hub.download_url_to_file(
        SAM_CHECKPOINT_URL,
        str(SAM_CHECKPOINT_PATH),
        progress=True,
    )
    print("Download complete.")
else:
    print(f"Using cached SAM checkpoint at {SAM_CHECKPOINT_PATH}")

# Build the model and wrap it in a predictor. The .to("cuda") move
# puts the model on GPU so the per-image inference in 3.2 runs in a
# few hundred milliseconds rather than tens of seconds.
sam = sam_model_registry["vit_h"](checkpoint=str(SAM_CHECKPOINT_PATH))
sam.to("cuda")
predictor = SamPredictor(sam)

print(f"SAM vit_h loaded on {next(sam.parameters()).device}")

In [ ]:
### 2.4) Define segment_leaf

import numpy as np


def segment_leaf(image_pil):
    """Run SAM on a PIL image and return the predicted leaf mask.

    Uses the precommit's prompting strategy: a single foreground point
    at the image center. SAM returns three candidate masks at different
    nested scales; this function takes the one SAM scores highest.

    Parameters
    ----------
    image_pil : PIL.Image
        The source image. RGB conversion is applied defensively.

    Returns
    -------
    mask : numpy.ndarray
        Boolean mask, shape (H, W), True where SAM identifies the leaf.
    """
    image_pil = image_pil.convert("RGB")
    image_np = np.array(image_pil)
    H, W = image_np.shape[:2]

    # set_image preprocesses the image (resize, normalize) and runs
    # the SAM image encoder. This is the expensive step (~100 ms on
    # T4); everything after is cheap.
    predictor.set_image(image_np)

    # Single foreground point at image center.
    center_point = np.array([[W // 2, H // 2]])
    foreground_label = np.array([1])  # 1 = foreground, 0 = background

    # multimask_output=True returns three candidate masks at different
    # nested scales (small, medium, large) along with SAM's own quality
    # score for each. The highest-scoring one is the model's most
    # confident leaf boundary for this prompt.
    masks, scores, _ = predictor.predict(
        point_coords=center_point,
        point_labels=foreground_label,
        multimask_output=True,
    )

    best_idx = int(np.argmax(scores))
    return masks[best_idx]

In [ ]:
### 2.5) Smoke-test segment_leaf on the first working-set image

import matplotlib.pyplot as plt

test_filename = working_set["filename"].iloc[0]
test_image = load_pd_image(test_filename)
test_mask = segment_leaf(test_image)

# Verify the mask shape and dtype match expectations before plotting.
test_image_np = np.array(test_image)
assert test_mask.shape == test_image_np.shape[:2], (
    f"Mask shape {test_mask.shape} doesn't match image shape "
    f"{test_image_np.shape[:2]}"
)
assert test_mask.dtype == bool, f"Expected boolean mask, got {test_mask.dtype}"

# Show the image and the mask side by side.
fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(test_image_np)
axes[0].set_title(f"Image: {test_filename}")
axes[0].axis("off")
axes[1].imshow(test_mask, cmap="gray")
axes[1].set_title(f"SAM mask ({test_mask.mean():.1%} of pixels)")
axes[1].axis("off")
plt.tight_layout()
plt.show()

print(
    f"Smoke test passed — mask shape {test_mask.shape}, "
    f"{test_mask.mean():.1%} foreground"
)

## 3) Run SAM on the full working set

This subsection runs `segment_leaf` on each of the 81 working-set
images and saves the result as a PNG. The per-image saves happen
inside the loop (not batched at the end), so a kernel crash partway
through doesn't lose progress — masks already on disk stay there.

What gets saved:

- 81 PNGs in `OUTPUT_DIR / "leaf_masks"`, one per working-set image,
  named `<filename>.png` to match the working set's `filename`
  column. Mode is grayscale (0 = not leaf, 255 = leaf), so any image
  viewer can open them for visual spot-checking.
- A `leaf_masks_metadata.parquet` recording the file path and
  foreground fraction (the share of pixels SAM marked as leaf) for
  each image. Subsections 2.3 and 2.4 add columns to this table.

Total runtime is typically around a minute on T4 — SAM inference is
a few hundred milliseconds per image, with the rest spent on file
I/O to Drive.

In [ ]:
### 3.1) Segment every working-set image

from tqdm.auto import tqdm
from PIL import Image
import time

mask_dir = OUTPUT_DIR / "leaf_masks"
mask_dir.mkdir(exist_ok=True)

# Track per-image stats so we can persist them in the metadata parquet
# and surface a distribution plot in the next cell.
per_image_records = []

start_time = time.time()
for _, row in tqdm(working_set.iterrows(), total=len(working_set), desc="Segmenting"):
    filename = row["filename"]
    image_pil = load_pd_image(filename)
    mask = segment_leaf(image_pil)

    # Save as grayscale PNG (0 = not leaf, 255 = leaf). Mode 'L' is
    # universally readable by any image viewer; after PNG compression
    # the file size is essentially the same as mode '1' (binary).
    mask_path = mask_dir / f"{filename}.png"
    Image.fromarray((mask * 255).astype(np.uint8), mode="L").save(mask_path)

    per_image_records.append({
        "filename": filename,
        "mask_path": str(mask_path),
        "fg_fraction": float(mask.mean()),
    })

elapsed = time.time() - start_time

# Persist the per-image metadata. 2.3 and 2.4 will reread and extend
# this table; Section 3 will reference mask_path to load each mask.
leaf_masks_metadata = pd.DataFrame(per_image_records)
metadata_path = OUTPUT_DIR / "leaf_masks_metadata.parquet"
leaf_masks_metadata.to_parquet(metadata_path)

print(f"\nSegmented {len(working_set)} images in {elapsed:.1f}s")
print(f"  ({elapsed / len(working_set):.2f}s per image average)")
print(f"  Masks saved to: {mask_dir}")
print(f"  Metadata saved to: {metadata_path}")

In [ ]:
### 3.2) Plot the distribution of foreground fractions

# A quick visual of how much of each image SAM marked as leaf. We
# expect a unimodal distribution centered somewhere in the 20–60%
# range with a tail toward smaller values (images where the leaf is
# a smaller subject against a lot of background). A bimodal or
# heavily skewed distribution would suggest SAM struggling on a
# subset of images.

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(
    leaf_masks_metadata["fg_fraction"],
    bins=20,
    edgecolor="black",
    linewidth=0.5,
)
ax.set_xlabel("Foreground fraction (share of pixels SAM marked as leaf)")
ax.set_ylabel("Number of images")
ax.set_title(
    "Distribution of SAM mask foreground fractions across "
    f"the {len(leaf_masks_metadata)} working-set images"
)
ax.set_xlim(0, 1)
plt.tight_layout()
plt.show()

# Summary stats.
fg_summary = leaf_masks_metadata["fg_fraction"].describe()
print("Foreground fraction summary:")
print(f"  median:  {fg_summary['50%']:.1%}")
print(f"  IQR:     {fg_summary['25%']:.1%} – {fg_summary['75%']:.1%}")
print(f"  min:     {fg_summary['min']:.1%}")
print(f"  max:     {fg_summary['max']:.1%}")

# Flag any images that look pathological — very small or very large
# masks. These don't fail the section, but they're worth eyeballing.
# The formal pass/fail decision is 2.3's IoU validation, not this.
EXTREME_LOW = 0.02   # below 2% — SAM probably found nothing
EXTREME_HIGH = 0.95  # above 95% — SAM probably found everything

extreme_low = leaf_masks_metadata[
    leaf_masks_metadata["fg_fraction"] < EXTREME_LOW
]
extreme_high = leaf_masks_metadata[
    leaf_masks_metadata["fg_fraction"] > EXTREME_HIGH
]

n_low = len(extreme_low)
n_high = len(extreme_high)

if n_low or n_high:
    print(f"\nExtreme masks worth eyeballing:")
    print(f"  {n_low} image(s) with fg_fraction < {EXTREME_LOW:.0%}")
    print(f"  {n_high} image(s) with fg_fraction > {EXTREME_HIGH:.0%}")
    print(
        f"  These don't fail Section 2 — 2.3's IoU check is the "
        f"formal validation — but extreme outliers can change "
        f"what 2.4 routes to the fallback."
    )
else:
    print(
        f"\nNo extreme masks "
        f"(none below {EXTREME_LOW:.0%} or above {EXTREME_HIGH:.0%})."
    )

## 4) Run SAM on the full working set

This subsection runs `segment_leaf` on each of the 81 working-set
images and saves the result as a PNG. The per-image saves happen
inside the loop (not batched at the end), so a kernel crash partway
through doesn't lose progress — masks already on disk stay there.

What gets saved:

- 81 PNGs in `OUTPUT_DIR / "leaf_masks"`, one per working-set image,
  named `<filename>.png` to match the working set's `filename`
  column. Mode is grayscale (0 = not leaf, 255 = leaf), so any image
  viewer can open them for visual spot-checking.
- A `leaf_masks_metadata.parquet` recording the file path and
  foreground fraction (the share of pixels SAM marked as leaf) for
  each image. Subsections 2.3 and 2.4 add columns to this table.

Total runtime is typically around a minute on T4 — SAM inference is
a few hundred milliseconds per image, with the rest spent on file
I/O to Drive.

In [ ]:
### 4.1) Segment every working-set image

from tqdm.auto import tqdm
from PIL import Image
import time

mask_dir = OUTPUT_DIR / "leaf_masks"
mask_dir.mkdir(exist_ok=True)

# Track per-image stats so we can persist them in the metadata parquet
# and surface a distribution plot in the next cell.
per_image_records = []

start_time = time.time()
for _, row in tqdm(working_set.iterrows(), total=len(working_set), desc="Segmenting"):
    filename = row["filename"]
    image_pil = load_pd_image(filename)
    mask = segment_leaf(image_pil)

    # Save as grayscale PNG (0 = not leaf, 255 = leaf). Mode 'L' is
    # universally readable by any image viewer; after PNG compression
    # the file size is essentially the same as mode '1' (binary).
    mask_path = mask_dir / f"{filename}.png"
    Image.fromarray((mask * 255).astype(np.uint8), mode="L").save(mask_path)

    per_image_records.append({
        "filename": filename,
        "mask_path": str(mask_path),
        "fg_fraction": float(mask.mean()),
    })

elapsed = time.time() - start_time

# Persist the per-image metadata. 2.3 and 2.4 will reread and extend
# this table; Section 3 will reference mask_path to load each mask.
leaf_masks_metadata = pd.DataFrame(per_image_records)
metadata_path = OUTPUT_DIR / "leaf_masks_metadata.parquet"
leaf_masks_metadata.to_parquet(metadata_path)

print(f"\nSegmented {len(working_set)} images in {elapsed:.1f}s")
print(f"  ({elapsed / len(working_set):.2f}s per image average)")
print(f"  Masks saved to: {mask_dir}")
print(f"  Metadata saved to: {metadata_path}")

In [ ]:
### 4.2) Plot the distribution of foreground fractions

# A quick visual of how much of each image SAM marked as leaf. We
# expect a unimodal distribution centered somewhere in the 20–60%
# range with a tail toward smaller values (images where the leaf is
# a smaller subject against a lot of background). A bimodal or
# heavily skewed distribution would suggest SAM struggling on a
# subset of images.

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(
    leaf_masks_metadata["fg_fraction"],
    bins=20,
    edgecolor="black",
    linewidth=0.5,
)
ax.set_xlabel("Foreground fraction (share of pixels SAM marked as leaf)")
ax.set_ylabel("Number of images")
ax.set_title(
    "Distribution of SAM mask foreground fractions across "
    f"the {len(leaf_masks_metadata)} working-set images"
)
ax.set_xlim(0, 1)
plt.tight_layout()
plt.show()

# Summary stats.
fg_summary = leaf_masks_metadata["fg_fraction"].describe()
print("Foreground fraction summary:")
print(f"  median:  {fg_summary['50%']:.1%}")
print(f"  IQR:     {fg_summary['25%']:.1%} – {fg_summary['75%']:.1%}")
print(f"  min:     {fg_summary['min']:.1%}")
print(f"  max:     {fg_summary['max']:.1%}")

# Flag any images that look pathological — very small or very large
# masks. These don't fail the section, but they're worth eyeballing.
# The formal pass/fail decision is 2.3's IoU validation, not this.
EXTREME_LOW = 0.02   # below 2% — SAM probably found nothing
EXTREME_HIGH = 0.95  # above 95% — SAM probably found everything

extreme_low = leaf_masks_metadata[
    leaf_masks_metadata["fg_fraction"] < EXTREME_LOW
]
extreme_high = leaf_masks_metadata[
    leaf_masks_metadata["fg_fraction"] > EXTREME_HIGH
]

n_low = len(extreme_low)
n_high = len(extreme_high)

if n_low or n_high:
    print(f"\nExtreme masks worth eyeballing:")
    print(f"  {n_low} image(s) with fg_fraction < {EXTREME_LOW:.0%}")
    print(f"  {n_high} image(s) with fg_fraction > {EXTREME_HIGH:.0%}")
    print(
        f"  These don't fail Section 2 — 2.3's IoU check is the "
        f"formal validation — but extreme outliers can change "
        f"what 2.4 routes to the fallback."
    )
else:
    print(
        f"\nNo extreme masks "
        f"(none below {EXTREME_LOW:.0%} or above {EXTREME_HIGH:.0%})."
    )